# 07. 직능원 내 자리 PC 3차 학습률 조정 재현 실험

## 목적
직능원 교실 **내 자리 PC에서 완료한 2차 재현 학습의 Best Accuracy 체크포인트**를 시작점으로 불러온 뒤,
`learning rate = 0.0001`로 낮춰 MobileNetV3 Small, EfficientNetB0, ResNet18 세 모델을 추가 학습한다.

이번 실험의 핵심 목적은 **최고 성능을 새로 찾는 것보다, 앞서 수행한 학습률 조정 효과가 내 자리 PC에서도 비슷하게 재현되는지 확인하는 것**이다.

### 기존 직능원 교실 PC 3차와 동일하게 유지하는 조건
- Training: TRAIN_01 + TRAIN_02 = **223,578장**
- Validation: **52,126장**
- 입력: 224×224 grayscale PNG → 3채널 복제
- ImageNet Normalize 적용
- Batch Size = **32**
- Optimizer = **Adam**
- Loss = **CrossEntropyLoss**
- Learning Rate = **0.0001**
- Max Epoch = **5**
- EarlyStopping patience = **3**
- EarlyStopping 기준 = **Validation Loss**
- Best Validation Loss / Best Validation Accuracy 체크포인트 각각 저장
- seed = **42**
- num_workers = **0**
- Scheduler 사용 안 함
- Weight Decay 사용 안 함

### 이번 3차에서 달라지는 점
1. 시작 가중치는 **직능원 내 자리 PC 2차 Best Accuracy 모델**
2. Learning Rate를 2차의 `0.001`에서 `0.0001`로 낮춤
3. 기존 2차/기존 3차 파일을 덮어쓰지 않도록 별도 실험 태그로 저장

> 주의: 2차에서 저장한 것은 `model.state_dict()`이므로 Adam optimizer 상태는 이어받지 않는다.  
> 3차에서는 불러온 모델 가중치에 대해 **새 Adam optimizer(lr=0.0001)** 를 생성한다.

### 비교 흐름
`직능원 내 자리 PC 2차 Best Accuracy`
→ `같은 PC에서 lr=0.0001로 3차 학습률 조정`
→ `직능원 내 자리 PC 3차 Best Accuracy`

이 결과를 앞서 수행한 **집 PC 2차 → 직능원 교실 PC 3차** 흐름과 비교하여
학습률 조정에 따른 성능 향상 경향이 다시 나타나는지 확인한다.


# 1. 학습 환경 및 경로 설정
기존 프로젝트 폴더 구조와 2차 학습 결과를 그대로 사용한다.
먼저 GPU, 데이터, 2차 Best Accuracy 체크포인트가 모두 존재하는지 확인한다.

In [ ]:
from pathlib import Path
import json
import random
import time
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

# --------------------------------------------------
# 재현성용 seed
# --------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------------
# 프로젝트 기본 경로
# --------------------------------------------------
PROJECT_ROOT = Path(r"D:\emotion_recognition_project")

GRAY_TRAIN_DIR = PROJECT_ROOT / "02_data" / "processed" / "grayscale_png" / "train"
GRAY_VALID_DIR = PROJECT_ROOT / "02_data" / "processed" / "grayscale_png" / "valid"

TRAIN_LABEL_DIR = PROJECT_ROOT / "02_data" / "labels" / "train"
VALID_LABEL_DIR = PROJECT_ROOT / "02_data" / "labels" / "valid"

MODEL_DIR = PROJECT_ROOT / "05_models"
OUTPUT_DIR = PROJECT_ROOT / "06_outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 7개 감정 클래스 순서
# 2차 학습과 반드시 동일해야 한다.
# --------------------------------------------------
CLASS_NAMES = ["기쁨", "당황", "분노", "불안", "상처", "슬픔", "중립"]
LABEL_TO_IDX = {label: idx for idx, label in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

# --------------------------------------------------
# 3차 공통 학습 조건
# 2차와 동일, Learning Rate만 0.0001
# --------------------------------------------------
BATCH_SIZE = 32
LEARNING_RATE = 0.0001
MAX_EPOCHS = 5
PATIENCE = 3
NUM_WORKERS = 0

EXPERIMENT_TAG = "v3_repro_institute_seatpc_lr_adjust_short5"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
print("사용 장치:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("감정 클래스:", CLASS_NAMES)
print("Learning Rate:", LEARNING_RATE)
print("Batch Size:", BATCH_SIZE)
print("Max Epoch:", MAX_EPOCHS)
print("EarlyStopping patience:", PATIENCE)

## 1-1. 직능원 내 자리 PC 2차 Best Accuracy 체크포인트 경로 확인

직능원 내 자리 PC에서 완료한 2차 재현 학습의 세 모델 `Best Accuracy` 체크포인트가
`05_models` 폴더에 있어야 한다.

이번 3차 학습은 반드시 아래 세 파일에서 시작한다.

- `mobilenet_v3_small_pretrained_v2_repro_institute_seatpc_best_accuracy.pt`
- `efficientnet_b0_pretrained_v2_repro_institute_seatpc_best_accuracy.pt`
- `resnet18_pretrained_v2_repro_institute_seatpc_best_accuracy.pt`

하나라도 없으면 학습을 시작하지 않고 파일 경로를 먼저 확인한다.


In [ ]:
V2_BEST_ACCURACY_PATHS = {
    "mobilenet_v3_small": (
        MODEL_DIR
        / "mobilenet_v3_small_pretrained_v2_repro_institute_seatpc_best_accuracy.pt"
    ),
    "efficientnet_b0": (
        MODEL_DIR
        / "efficientnet_b0_pretrained_v2_repro_institute_seatpc_best_accuracy.pt"
    ),
    "resnet18": (
        MODEL_DIR
        / "resnet18_pretrained_v2_repro_institute_seatpc_best_accuracy.pt"
    ),
}

all_checkpoints_ok = True

for model_name, checkpoint_path in V2_BEST_ACCURACY_PATHS.items():
    exists = checkpoint_path.exists()

    print(
        f"{model_name:20s} | "
        f"존재={exists} | "
        f"{checkpoint_path}"
    )

    if not exists:
        all_checkpoints_ok = False

assert all_checkpoints_ok, (
    "직능원 내 자리 PC 2차 Best Accuracy 체크포인트 중 "
    "누락된 파일이 있습니다. "
    "05_models 폴더의 파일명을 먼저 확인하세요."
)

print(
    "\n✅ 직능원 내 자리 PC의 세 모델 "
    "2차 Best Accuracy 체크포인트 확인 완료"
)


# 2. grayscale PNG와 JSON 라벨 연결
2차와 동일하게 PNG 파일명과 JSON의 `filename`을 연결하고,
정답 라벨은 `faceExp_uploader`를 사용한다.

In [ ]:
def build_dataframe(image_dir, label_dir):
    """
    전처리된 grayscale PNG와 AI-Hub JSON 라벨을 filename(stem) 기준으로 연결한다.
    """
    image_map = {
        path.stem: path
        for path in image_dir.glob("*.png")
    }

    records = []

    for json_path in sorted(label_dir.rglob("*.json")):
        with open(json_path, "r", encoding="utf-8") as f:
            label_data = json.load(f)

        for record in label_data:
            original_name = record["filename"]
            stem = Path(original_name).stem
            label = record["faceExp_uploader"]

            if stem in image_map and label in LABEL_TO_IDX:
                records.append({
                    "filename": original_name,
                    "image_path": str(image_map[stem]),
                    "label": label,
                    "label_index": LABEL_TO_IDX[label],
                })

    return pd.DataFrame(records)


train_df = build_dataframe(GRAY_TRAIN_DIR, TRAIN_LABEL_DIR)
valid_df = build_dataframe(GRAY_VALID_DIR, VALID_LABEL_DIR)

print("Training 연결 수:", len(train_df))
print("Validation 연결 수:", len(valid_df))

# 프로젝트에서 확정한 실제 학습 데이터 수와 다르면 즉시 중단
assert len(train_df) == 223_578, f"Training 연결 수 오류: {len(train_df)}"
assert len(valid_df) == 52_126, f"Validation 연결 수 오류: {len(valid_df)}"

print("\nTraining 클래스별 개수")
print(train_df["label"].value_counts().reindex(CLASS_NAMES))

print("\nValidation 클래스별 개수")
print(valid_df["label"].value_counts().reindex(CLASS_NAMES))

print("\n✅ 데이터 연결 검증 완료")

# 3. Dataset / DataLoader 구성
grayscale 이미지를 3채널로 복제한 뒤 ImageNet Normalize를 적용한다.
2차 pretrained 학습과 같은 입력 방식이다.

In [ ]:
# --------------------------------------------------
# 2차와 동일한 입력 변환
# --------------------------------------------------
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class EmotionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 저장된 grayscale PNG를 PIL로 불러온다.
        image = Image.open(row["image_path"]).convert("L")

        if self.transform is not None:
            image = self.transform(image)

        label = int(row["label_index"])
        return image, label


train_dataset = EmotionDataset(train_df, transform=image_transform)
valid_dataset = EmotionDataset(valid_df, transform=image_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("train_dataset:", len(train_dataset))
print("valid_dataset:", len(valid_dataset))
print("Training batch 수:", len(train_loader))
print("Validation batch 수:", len(valid_loader))

sample_images, sample_labels = next(iter(train_loader))
print("Batch image shape:", sample_images.shape)
print("Batch label shape:", sample_labels.shape)
print("Label min/max:", sample_labels.min().item(), sample_labels.max().item())

assert len(train_dataset) == 223_578
assert len(valid_dataset) == 52_126
assert sample_images.shape[1:] == (3, 224, 224)
assert sample_labels.min().item() >= 0
assert sample_labels.max().item() <= 6

print("\n✅ Dataset / DataLoader 검증 완료")

# 4. 모델 구조 생성 및 2차 가중치 불러오기

중요한 점:
- 여기서는 `weights=None`으로 **모델 구조만 생성**한다.
- 그 다음 2차 Best Accuracy `state_dict`를 불러온다.
- 즉 3차의 실제 시작 가중치는 random이 아니라 **2차 학습 완료 모델**이다.

In [ ]:
def create_model(model_name, num_classes=NUM_CLASSES):
    """
    2차와 동일한 torchvision 모델 구조를 만든다.
    pretrained 다운로드는 하지 않고, 바로 2차 state_dict를 불러온다.
    """
    if model_name == "mobilenet_v3_small":
        model = models.mobilenet_v3_small(weights=None)
        model.classifier[3] = nn.Linear(
            model.classifier[3].in_features,
            num_classes,
        )

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes,
        )

    elif model_name == "resnet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes,
        )

    else:
        raise ValueError(f"지원하지 않는 모델명: {model_name}")

    return model


def load_v2_best_accuracy_model(model_name):
    """
    모델 구조 생성 → 2차 Best Accuracy 가중치 로드 → GPU 이동
    """
    checkpoint_path = V2_BEST_ACCURACY_PATHS[model_name]

    model = create_model(model_name)
    state_dict = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )
    model.load_state_dict(state_dict)
    model = model.to(device)

    print(f"✅ {model_name} 직능원 내 자리 PC 2차 Best Accuracy 가중치 로드 완료")
    print("   ", checkpoint_path)

    return model

# 5. 1 Epoch 학습 / Validation 함수
Training에서만 `backward()`와 `optimizer.step()`을 수행한다.
Validation에서는 가중치를 수정하지 않는다.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # 이전 batch의 gradient 초기화
        optimizer.zero_grad()

        # 순전파
        outputs = model(images)
        loss = criterion(outputs, labels)

        # 역전파 + 가중치 업데이트
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


@torch.no_grad()
def validate_one_epoch(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

# 6. 상세 Validation 평가 함수
Accuracy뿐 아니라 Precision / Recall / F1-score와 Confusion Matrix 계산에 사용한다.

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    all_labels = []
    all_preds = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)

        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        labels=list(range(NUM_CLASSES)),
        average=None,
        zero_division=0,
    )

    mean_precision = float(np.mean(precision))
    mean_recall = float(np.mean(recall))
    mean_f1 = float(np.mean(f1))

    cm = confusion_matrix(
        all_labels,
        all_preds,
        labels=list(range(NUM_CLASSES)),
    )

    return {
        "accuracy": float(accuracy),
        "mean_precision": mean_precision,
        "mean_recall": mean_recall,
        "mean_f1": mean_f1,
        "precision_per_class": precision,
        "recall_per_class": recall,
        "f1_per_class": f1,
        "confusion_matrix": cm,
    }

# 7. 3차 학습률 조정 함수

### 저장 기준
- Validation Loss가 가장 낮아질 때: `best_loss.pt`
- Validation Accuracy가 가장 높아질 때: `best_accuracy.pt`
- EarlyStopping은 **Validation Loss 기준, patience=3**

### 핵심
- 시작점: **직능원 내 자리 PC 2차 Best Accuracy**
- Learning Rate: **0.0001**
- 최대 Epoch: **5**
- 기존 직능원 교실 PC 3차와 동일한 조건으로 실행

2차 체크포인트와 기존 3차 체크포인트는 절대 덮어쓰지 않는다.


In [ ]:
def train_with_adjusted_lr(
    model_name,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    learning_rate=LEARNING_RATE,
):
    print("=" * 80)
    print(f"3차 학습률 조정 학습 시작: {model_name}")
    print(f"시작 체크포인트: {V2_BEST_ACCURACY_PATHS[model_name].name}")
    print(f"Learning Rate: {learning_rate}")
    print("=" * 80)

    # --------------------------------------------------
    # 2차 Best Accuracy 모델에서 시작
    # --------------------------------------------------
    model = load_v2_best_accuracy_model(model_name)

    criterion = nn.CrossEntropyLoss()

    # 중요:
    # 2차 optimizer 상태는 저장하지 않았으므로,
    # 3차에서는 낮은 LR의 새 Adam optimizer를 사용한다.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    # --------------------------------------------------
    # 3차 전용 저장 경로
    # --------------------------------------------------
    best_loss_path = (
        MODEL_DIR /
        f"{model_name}_pretrained_{EXPERIMENT_TAG}_best_loss.pt"
    )

    best_accuracy_path = (
        MODEL_DIR /
        f"{model_name}_pretrained_{EXPERIMENT_TAG}_best_accuracy.pt"
    )

    history_path = (
        OUTPUT_DIR /
        f"{model_name}_pretrained_{EXPERIMENT_TAG}_history.csv"
    )

    # --------------------------------------------------
    # Best 값 초기화
    # --------------------------------------------------
    best_val_loss = float("inf")
    best_val_acc = -1.0

    best_loss_epoch = None
    best_accuracy_epoch = None

    no_improve_count = 0
    history = []

    total_start = time.time()

    for epoch in range(1, max_epochs + 1):
        epoch_start = time.time()

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
        )

        valid_loss, valid_acc = validate_one_epoch(
            model,
            valid_loader,
            criterion,
        )

        epoch_seconds = time.time() - epoch_start

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "valid_loss": valid_loss,
            "valid_accuracy": valid_acc,
            "epoch_seconds": epoch_seconds,
        })

        print(
            f"Epoch {epoch:02d}/{max_epochs} | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Acc {train_acc:.4f} | "
            f"Val Loss {valid_loss:.4f} | "
            f"Val Acc {valid_acc:.4f} | "
            f"{epoch_seconds:.1f}s"
        )

        # --------------------------------------------------
        # Best Validation Loss 저장
        # EarlyStopping 카운트도 Loss 기준
        # --------------------------------------------------
        if valid_loss < best_val_loss:
            best_val_loss = valid_loss
            best_loss_epoch = epoch
            no_improve_count = 0

            torch.save(
                model.state_dict(),
                best_loss_path,
            )

            print(
                f"  ✅ Best Loss 저장 | "
                f"Epoch {epoch} | Val Loss {valid_loss:.4f}"
            )
        else:
            no_improve_count += 1
            print(
                f"  Loss 개선 없음: "
                f"{no_improve_count}/{patience}"
            )

        # --------------------------------------------------
        # Best Validation Accuracy 별도 저장
        # --------------------------------------------------
        if valid_acc > best_val_acc:
            best_val_acc = valid_acc
            best_accuracy_epoch = epoch

            torch.save(
                model.state_dict(),
                best_accuracy_path,
            )

            print(
                f"  ✅ Best Accuracy 저장 | "
                f"Epoch {epoch} | Val Acc {valid_acc:.4f}"
            )

        # --------------------------------------------------
        # 매 Epoch history CSV 갱신
        # 학습 중 중단되어도 진행 기록 보존
        # --------------------------------------------------
        pd.DataFrame(history).to_csv(
            history_path,
            index=False,
            encoding="utf-8-sig",
        )

        # --------------------------------------------------
        # EarlyStopping
        # --------------------------------------------------
        if no_improve_count >= patience:
            print(
                f"\n⏹ EarlyStopping: "
                f"Validation Loss가 {patience} Epoch 연속 개선되지 않음"
            )
            break

    total_seconds = time.time() - total_start

    print("\n" + "-" * 80)
    print(f"{model_name} 3차 학습률 조정 학습 완료")
    print(f"Best Loss Epoch: {best_loss_epoch}")
    print(f"Best Validation Loss: {best_val_loss:.6f}")
    print(f"Best Accuracy Epoch: {best_accuracy_epoch}")
    print(f"Best Validation Accuracy: {best_val_acc:.6f}")
    print(f"총 학습 시간: {total_seconds / 60:.1f}분")
    print("Best Loss 모델:", best_loss_path)
    print("Best Accuracy 모델:", best_accuracy_path)
    print("History:", history_path)
    print("-" * 80)

    return {
        "model_name": model_name,
        "best_loss_epoch": best_loss_epoch,
        "best_val_loss": best_val_loss,
        "best_accuracy_epoch": best_accuracy_epoch,
        "best_val_accuracy": best_val_acc,
        "best_loss_path": best_loss_path,
        "best_accuracy_path": best_accuracy_path,
        "history_path": history_path,
        "history": pd.DataFrame(history),
        "total_seconds": total_seconds,
    }

# 8. 3차 학습 전 2차 Best Accuracy 기준 성능 재확인
직능원 직능원 교실 PC에서 체크포인트가 정상 로드되고,
집에서 얻은 결과와 비슷한 Validation 성능이 나오는지 먼저 확인한다.

이 셀은 **학습하지 않고 평가만** 한다.

In [ ]:
baseline_rows = []

for model_name in V2_BEST_ACCURACY_PATHS:
    print("\n" + "=" * 80)
    print("2차 체크포인트 재평가:", model_name)

    baseline_model = load_v2_best_accuracy_model(model_name)

    criterion = nn.CrossEntropyLoss()
    baseline_loss, baseline_acc = validate_one_epoch(
        baseline_model,
        valid_loader,
        criterion,
    )
    baseline_metrics = evaluate_model(
        baseline_model,
        valid_loader,
    )

    baseline_rows.append({
        "model": model_name,
        "stage": "v2_best_accuracy_baseline",
        "valid_loss": baseline_loss,
        "accuracy": baseline_metrics["accuracy"],
        "mean_precision": baseline_metrics["mean_precision"],
        "mean_recall": baseline_metrics["mean_recall"],
        "mean_f1": baseline_metrics["mean_f1"],
    })

    print(f"Validation Loss: {baseline_loss:.6f}")
    print(f"Validation Accuracy: {baseline_metrics['accuracy']:.4%}")
    print(f"Mean F1: {baseline_metrics['mean_f1']:.4%}")

    del baseline_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

baseline_df = pd.DataFrame(baseline_rows)
baseline_df

# 9. MobileNetV3 Small 3차 학습률 조정
2차 Best Accuracy 체크포인트에서 시작해 lr=0.0001로 추가 학습한다.

In [ ]:
mobilenet_v3_result = train_with_adjusted_lr(
    "mobilenet_v3_small"
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 10. EfficientNetB0 3차 학습률 조정
2차 Best Accuracy 체크포인트에서 시작해 lr=0.0001로 추가 학습한다.

In [ ]:
efficientnet_b0_result = train_with_adjusted_lr(
    "efficientnet_b0"
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 11. ResNet18 3차 학습률 조정
2차 Best Accuracy 체크포인트에서 시작해 lr=0.0001로 추가 학습한다.

In [ ]:
resnet18_result = train_with_adjusted_lr(
    "resnet18"
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 12. 직능원 내 자리 PC 3차 Best Accuracy 체크포인트 최종 평가

이미 학습이 끝난 **직능원 교실 PC의 3차 학습률 조정 Best Accuracy 체크포인트**를 불러와 Validation 52,126장으로 평가한다.

이번 실험의 중심 질문은 다음과 같다.

> **직능원 내 자리 PC에서 학습한 2차 Best Accuracy 모델을 시작점으로 사용하고, 학습률을 0.001 → 0.0001로 낮춘 뒤 성능이 향상되었는가?**

따라서 이 단계에서는 `Best Loss`를 비교하지 않고 **3차 Best Accuracy만 평가**한다.


In [ ]:
# 3차 학습 결과는 다시 학습하지 않고 저장된 Best Accuracy 가중치를 불러와 평가한다.
v3_best_accuracy_paths = {
    model_name: MODEL_DIR / f"{model_name}_pretrained_{EXPERIMENT_TAG}_best_accuracy.pt"
    for model_name in V2_BEST_ACCURACY_PATHS
}

v3_evaluation_rows = []
v3_per_class_rows = []

for model_name, checkpoint_path in v3_best_accuracy_paths.items():
    print("\n" + "=" * 80)
    print("직능원 내 자리 PC 3차 Best Accuracy 평가:", model_name)
    print("불러오는 체크포인트:", checkpoint_path)

    model = create_model(model_name).to(device)
    state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]
    model.load_state_dict(state_dict)

    criterion = nn.CrossEntropyLoss()
    valid_loss, _ = validate_one_epoch(model, valid_loader, criterion)
    metrics = evaluate_model(model, valid_loader)

    v3_evaluation_rows.append({
        "model": model_name,
        "source": "직능원 내 자리 PC 3차 학습률 조정 Best Accuracy",
        "valid_loss": valid_loss,
        "accuracy": metrics["accuracy"],
        "mean_precision": metrics["mean_precision"],
        "mean_recall": metrics["mean_recall"],
        "mean_f1": metrics["mean_f1"],
    })

    for idx, class_name in enumerate(CLASS_NAMES):
        v3_per_class_rows.append({
            "model": model_name,
            "source": "직능원 내 자리 PC 3차 학습률 조정 Best Accuracy",
            "class_name": class_name,
            "precision": metrics["precision_per_class"][idx],
            "recall": metrics["recall_per_class"][idx],
            "f1": metrics["f1_per_class"][idx],
        })

    cm = metrics["confusion_matrix"]
    plt.figure(figsize=(9, 8))
    plt.imshow(cm)
    plt.title(f"{model_name} - V3 Best Accuracy Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45)
    plt.yticks(range(NUM_CLASSES), CLASS_NAMES)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    plt.tight_layout()
    cm_path = OUTPUT_DIR / f"{model_name}_pretrained_{EXPERIMENT_TAG}_best_accuracy_confusion_matrix.png"
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Validation Loss: {valid_loss:.6f}")
    print(f"Validation Accuracy: {metrics['accuracy']:.4%}")
    print(f"Mean F1: {metrics['mean_f1']:.4%}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

v3_eval_df = pd.DataFrame(v3_evaluation_rows)
v3_per_class_df = pd.DataFrame(v3_per_class_rows)

v3_eval_path = OUTPUT_DIR / f"pretrained_{EXPERIMENT_TAG}_best_accuracy_metrics.csv"
v3_per_class_path = OUTPUT_DIR / f"pretrained_{EXPERIMENT_TAG}_best_accuracy_per_class_metrics.csv"

v3_eval_df.to_csv(v3_eval_path, index=False, encoding="utf-8-sig")
v3_per_class_df.to_csv(v3_per_class_path, index=False, encoding="utf-8-sig")

print("\n✅ 직능원 내 자리 PC 3차 Best Accuracy 평가 완료")
display(v3_eval_df.sort_values(["accuracy", "mean_f1"], ascending=False))


# 13. 직능원 내 자리 PC 2차 Best Accuracy vs 직능원 내 자리 PC 3차 Best Accuracy 비교

여기서는 **어느 결과가 어느 PC에서 만들어진 것인지 명확하게 표시**한다.

- **직능원 내 자리 PC 2차**: 집에서 학습해 저장한 `Best Accuracy` 체크포인트
- **직능원 내 자리 PC 3차**: 위 2차 체크포인트를 시작점으로, 직능원 교실 PC에서 learning rate를 `0.001 → 0.0001`로 낮춰 추가 학습한 `Best Accuracy` 체크포인트

주의할 점은, 아래 수치는 두 저장본을 **직능원 교실 PC에서 동일한 Validation 52,126장으로 재평가한 값**이라는 것이다. 즉 2차 행은 **집에서 학습된 모델**, 3차 행은 **직능원 교실 PC에서 추가 학습된 모델**을 뜻한다.


In [ ]:
# 2차는 "집에서 학습한 Best Accuracy 체크포인트"를 불러와 다시 평가한다.
v2_rows = []

for model_name, checkpoint_path in V2_BEST_ACCURACY_PATHS.items():
    print("\n" + "=" * 80)
    print("직능원 내 자리 PC에서 학습한 2차 Best Accuracy 재평가:", model_name)
    print("불러오는 체크포인트:", checkpoint_path)

    model = load_v2_best_accuracy_model(model_name)
    criterion = nn.CrossEntropyLoss()
    valid_loss, _ = validate_one_epoch(model, valid_loader, criterion)
    metrics = evaluate_model(model, valid_loader)

    v2_rows.append({
        "model": model_name,
        "source": "직능원 내 자리 PC 2차 Best Accuracy (동일 Validation 재평가)",
        "valid_loss": valid_loss,
        "accuracy": metrics["accuracy"],
        "mean_precision": metrics["mean_precision"],
        "mean_recall": metrics["mean_recall"],
        "mean_f1": metrics["mean_f1"],
    })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

v2_eval_df = pd.DataFrame(v2_rows)
comparison_df = pd.concat([v2_eval_df, v3_eval_df], ignore_index=True)

comparison_path = OUTPUT_DIR / f"institute_seatpc_v2_best_accuracy_vs_v3_{EXPERIMENT_TAG}_best_accuracy.csv"
comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")

improvement_rows = []
for model_name in V2_BEST_ACCURACY_PATHS:
    v2_row = v2_eval_df[v2_eval_df["model"] == model_name].iloc[0]
    v3_row = v3_eval_df[v3_eval_df["model"] == model_name].iloc[0]

    improvement_rows.append({
        "model": model_name,
        "내자리_PC_2차_Best_Accuracy": v2_row["accuracy"],
        "내자리_PC_3차_Best_Accuracy": v3_row["accuracy"],
        "Accuracy_변화_pp": (v3_row["accuracy"] - v2_row["accuracy"]) * 100,
        "내자리_PC_2차_Mean_F1": v2_row["mean_f1"],
        "내자리_PC_3차_Mean_F1": v3_row["mean_f1"],
        "Mean_F1_변화_pp": (v3_row["mean_f1"] - v2_row["mean_f1"]) * 100,
    })

improvement_df = pd.DataFrame(improvement_rows)
improvement_path = OUTPUT_DIR / f"institute_seatpc_v2_vs_v3_{EXPERIMENT_TAG}_best_accuracy_improvement.csv"
improvement_df.to_csv(improvement_path, index=False, encoding="utf-8-sig")

print("\n[표 1] 학습 출처가 표시된 2차 vs 3차 재평가 결과")
display(comparison_df.sort_values(["model", "source"]))

print("\n[표 2] 직능원 내 자리 PC 2차 → 내 자리 PC 3차 학습률 조정 후 성능 변화")
display(improvement_df)


# 14. 직능원 내 자리 PC 3차 학습 곡선 확인

3차 학습은 다시 실행하지 않는다. `06_outputs`에 이미 저장된 **직능원 내 자리 PC 3차 history CSV**를 불러와 그래프만 확인한다.

이 단계의 목적은 `Best Loss`와 `Best Accuracy`를 비교하는 것이 아니라,
- Train / Validation Loss 변화
- Train / Validation Accuracy 변화
- 과적합 경향
- EarlyStopping이 발생한 흐름
을 확인하는 것이다.


In [ ]:
v3_history_dfs = {}

for model_name in V2_BEST_ACCURACY_PATHS:
    history_path = OUTPUT_DIR / f"{model_name}_pretrained_{EXPERIMENT_TAG}_history.csv"
    history = pd.read_csv(history_path)
    v3_history_dfs[model_name] = history

    print("\n" + "=" * 80)
    print("직능원 내 자리 PC 3차 history 불러오기:", model_name)
    print(history_path)

    plt.figure(figsize=(8, 5))
    plt.plot(history["epoch"], history["train_loss"], label="Train Loss")
    plt.plot(history["epoch"], history["valid_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} - V3 Learning Rate Adjustment Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{model_name}_pretrained_{EXPERIMENT_TAG}_loss.png", dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history["epoch"], history["train_accuracy"], label="Train Accuracy")
    plt.plot(history["epoch"], history["valid_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} - V3 Learning Rate Adjustment Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{model_name}_pretrained_{EXPERIMENT_TAG}_accuracy.png", dpi=150, bbox_inches="tight")
    plt.show()

print("\n✅ 저장된 직능원 내 자리 PC 3차 history를 이용한 학습 곡선 생성 완료")


# 15. 최종 결과 해석

이번 비교의 출처와 흐름을 다음처럼 고정한다.

1. **직능원 내 자리 PC 2차 Best Accuracy**: 집 PC에서 학습해 저장한 체크포인트
2. **직능원 내 자리 PC 3차 Best Accuracy**: 위 2차 체크포인트를 불러와 직능원 교실 PC에서 learning rate를 `0.001 → 0.0001`로 낮춘 뒤 추가 학습한 체크포인트
3. 두 저장본은 직능원 교실 PC에서 동일한 Validation 52,126장으로 재평가해 비교
4. Accuracy와 Mean F1 변화량으로 학습률 조정 효과 확인
5. 3차 history로 과적합 / EarlyStopping 흐름 확인

이번 분석에서는 `Best Loss vs Best Accuracy` 비교를 하지 않는다. 또한 독립 Test set이 없으므로 결과는 **Validation Accuracy**라고 표현한다.


In [ ]:
final_summary_df = improvement_df.copy()
final_summary_df["Accuracy_향상"] = final_summary_df["Accuracy_변화_pp"] > 0
final_summary_df["Mean_F1_향상"] = final_summary_df["Mean_F1_변화_pp"] > 0

final_summary_path = OUTPUT_DIR / f"institute_seatpc_v2_to_v3_{EXPERIMENT_TAG}_final_summary.csv"
final_summary_df.to_csv(final_summary_path, index=False, encoding="utf-8-sig")

print("최종 요약 저장:", final_summary_path)
print("\n※ 직능원 내 자리 PC 2차 = 내 자리 PC에서 학습한 Best Accuracy 체크포인트")
print("※ 직능원 내 자리 PC 3차 = 위 체크포인트에서 lr=0.0001로 추가 학습한 Best Accuracy 체크포인트")
display(final_summary_df.sort_values("내자리_PC_3차_Best_Accuracy", ascending=False))
